# gVXR polychromatic package radiography

This reduced Colab experiment compares unfiltered and Al/Cu-filtered 160 kV spectra on a deterministic SiO₂/Cu/SAC305 package phantom. It validates spectral simulation and records reproducibility metadata; it is not a calibrated scanner model.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !apt-get update -qq
    !apt-get install -y -qq libnvidia-gl-580 libxcb-res0 libxcb-ewmh2 libxcb-composite0 libxcb-cursor0 libxcb-xinerama0 libxcb-keysyms1 libxcb-icccm4 libxcb-xkb1

repo = Path('/content/xsim-chip')
if repo.exists():
    !git -C {repo} pull --ff-only
else:
    !git clone -q https://github.com/yuweimin2077-hub/xsim-chip.git {repo}
%cd /content/xsim-chip
%pip install -q -e '.[spectral]'

In [ ]:
import json
import time
import matplotlib.pyplot as plt
import numpy as np
from gvxrPython3 import gvxr
from xsim_chip_analysis import (
    ConeBeamConfig, SimulationConfig, SpectrumConfig, build_run_manifest,
    normalise_energy_image, summarise_spectrum,
)

spectrum = SpectrumConfig()
geometry = ConeBeamConfig()
print('Core gVXR:', gvxr.getVersionOfCoreGVXR())
print('SimpleGVXR:', gvxr.getVersionOfSimpleGVXR())
print('Spectrum config:', json.dumps(spectrum.to_manifest(), indent=2))
print('Cone-beam config:', json.dumps(geometry.to_manifest(), indent=2))

## Compact multi-material package

The geometry uses non-overlapping built-in cuboids to avoid thousands of upstream STL components while preserving the three material classes and cone-beam magnification.

In [ ]:
started = time.perf_counter()
gvxr.createOpenGLContext()
gvxr.setSourcePosition(*geometry.source_position_mm, 'mm')
gvxr.usePointSource()
gvxr.setDetectorPosition(*geometry.detector_position_mm, 'mm')
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(*geometry.detector_pixels_xy)
gvxr.setDetectorPixelSize(*geometry.detector_pixel_size_mm, 'mm')

def add_cuboid(label, size_xyz_mm, position_xyz_mm):
    gvxr.makeCuboid(label, *size_xyz_mm, 'mm')
    gvxr.addPolygonMeshAsOuterSurface(label)
    gvxr.translateNode(label, *position_xyz_mm, 'mm')

add_cuboid('SiO2_core', (0.8, 8.0, 5.0), (0.0, 0.0, 0.0))
gvxr.setMixture('SiO2_core', [14, 8], [0.467, 0.533])
gvxr.setDensity('SiO2_core', 2.20, 'g/cm3')

for index, z_mm in enumerate((-1.4, 0.0, 1.4)):
    label = f'Cu_trace_{index}'
    add_cuboid(label, (0.12, 6.8, 0.18), (0.46, 0.0, z_mm))
    gvxr.setElement(label, 'Cu')

for index, y_mm in enumerate((-2.4, 0.0, 2.4)):
    label = f'SAC305_joint_{index}'
    add_cuboid(label, (0.55, 0.8, 0.8), (0.795, y_mm, -1.9))
    gvxr.setMixture(label, [50, 47, 29], [0.965, 0.030, 0.005])
    gvxr.setDensity(label, 7.38, 'g/cm3')

print('Package primitives created: 1 SiO2 core, 3 Cu traces, 3 SAC305 joints')

## Unfiltered versus filtered polychromatic exposure

gVXR generates the tube spectrum and applies material-dependent Beer–Lambert attenuation. The filtered exposure adds 0.5 mm Al inherent filtration and 1.0 mm Cu filtration.

In [ ]:
gvxr.setEnergyBinSize(spectrum.energy_bin_size_kev, 'keV')
gvxr.setVoltage(spectrum.tube_voltage_kv, 'kV')
gvxr.setmAs(spectrum.exposure_mas)

energy_unfiltered = np.asarray(gvxr.getEnergyBins('keV'), dtype=np.float32)
counts_unfiltered = np.asarray(gvxr.getPhotonCountsPerPixelAtSDD(), dtype=np.float64)
image_unfiltered = normalise_energy_image(
    np.asarray(gvxr.computeXRayImage()), gvxr.getTotalEnergyWithDetectorResponse()
)
summary_unfiltered = summarise_spectrum(energy_unfiltered, counts_unfiltered)

gvxr.addInherentFilter(spectrum.filters_mm[0][0], spectrum.filters_mm[0][1], 'mm')
gvxr.addFilter(spectrum.filters_mm[1][0], spectrum.filters_mm[1][1], 'mm')
energy_filtered = np.asarray(gvxr.getEnergyBins('keV'), dtype=np.float32)
counts_filtered = np.asarray(gvxr.getPhotonCountsPerPixelAtSDD(), dtype=np.float64)
image_filtered = normalise_energy_image(
    np.asarray(gvxr.computeXRayImage()), gvxr.getTotalEnergyWithDetectorResponse()
)
summary_filtered = summarise_spectrum(energy_filtered, counts_filtered)

object_mask = np.minimum(image_unfiltered, image_filtered) < 0.999
metrics = {
    'mean_energy_shift_kev': summary_filtered['mean_energy_kev'] - summary_unfiltered['mean_energy_kev'],
    'mean_unfiltered_transmission': float(image_unfiltered[object_mask].mean()),
    'mean_filtered_transmission': float(image_filtered[object_mask].mean()),
    'mean_absolute_transmission_change': float(np.abs(image_filtered - image_unfiltered)[object_mask].mean()),
    'object_pixels': int(object_mask.sum()),
}
print('Unfiltered:', json.dumps(summary_unfiltered, indent=2))
print('Filtered:', json.dumps(summary_filtered, indent=2))
print('Comparison:', json.dumps(metrics, indent=2))

In [ ]:
probability_unfiltered = counts_unfiltered / counts_unfiltered.sum()
probability_filtered = counts_filtered / counts_filtered.sum()
figure, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].bar(energy_unfiltered, probability_unfiltered, width=spectrum.energy_bin_size_kev, alpha=0.65, label='Unfiltered')
axes[0, 0].bar(energy_filtered, probability_filtered, width=spectrum.energy_bin_size_kev, alpha=0.65, label='0.5 mm Al + 1.0 mm Cu')
axes[0, 0].set(xlabel='Energy (keV)', ylabel='Photon probability', title='160 kV spectra')
axes[0, 0].legend()
axes[0, 1].imshow(image_unfiltered, cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Unfiltered transmission')
axes[1, 0].imshow(image_filtered, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title('Filtered transmission')
difference = image_filtered - image_unfiltered
limit = max(float(np.abs(difference).max()), 1e-6)
axes[1, 1].imshow(difference, cmap='coolwarm', vmin=-limit, vmax=limit)
axes[1, 1].set_title('Filtered − unfiltered')
for ax in (axes[0, 1], axes[1, 0], axes[1, 1]): ax.axis('off')
plt.tight_layout()

## Save reproducibility artefacts

The compact arrays and manifest are written outside Git. The manifest distinguishes measured outputs from scanner assumptions and limitations.

In [ ]:
elapsed = time.perf_counter() - started
output_dir = Path('/content/xsim_outputs/gvxr_spectral')
output_dir.mkdir(parents=True, exist_ok=True)
manifest = build_run_manifest(SimulationConfig.quick_colab(), elapsed_seconds=elapsed)
manifest['experiment'] = 'gvxr-polychromatic-radiography-v1'
manifest['spectrum'] = spectrum.to_manifest()
manifest['geometry'] = geometry.to_manifest()
manifest['gvxr_runtime'] = {
    'core': gvxr.getVersionOfCoreGVXR(),
    'simple': gvxr.getVersionOfSimpleGVXR(),
}
manifest['result'] = {
    'unfiltered_spectrum': summary_unfiltered,
    'filtered_spectrum': summary_filtered,
    **metrics,
}
manifest['limitations'] = [
    'Compact cuboid phantom, not the full NIST STL package',
    'Ideal energy-integrating detector; no scatter or detector blur',
    'Geometry and exposure are simulation assumptions, not scanner calibration',
]
(output_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
np.savez_compressed(
    output_dir / 'spectral_radiography.npz',
    energy_unfiltered_kev=energy_unfiltered, counts_unfiltered=counts_unfiltered,
    energy_filtered_kev=energy_filtered, counts_filtered=counts_filtered,
    image_unfiltered=image_unfiltered, image_filtered=image_filtered,
)
figure.savefig(output_dir / 'spectral_comparison.png', dpi=160, bbox_inches='tight')
gvxr.terminate()
print(json.dumps(manifest, indent=2))
print('Saved to', output_dir)